In [ ]:
function range(start: number, stop: number): number[] {
    return Array.from({ length: stop - start + 1 }, (_, index) => start + index);
}

In [ ]:
import { CSP, solve, Assignment, Variable, Formula } from "./02-Backtracking-Constraint-Solver";

# Saving the Goat

An agricultural economist has to sell a *wolf*, a *goat*, and a *cabbage* on a market place.  In order to
reach the market place, she has to cross a river.  The boat that she can use is so small that it can
only accommodate either the goat, the wolf, or the cabbage in addition to the agricultural economist.
Now if the agricultural economist leaves the wolf alone with the goat, the wolf will eat the goat.
If, instead, the agricultural economist leaves the goat with the cabbage, the goat will eat the cabbage.
Is it possible for the agricultural economist to develop a schedule that allows her to cross the river
without either the goat or the cabbage being eaten?

We will encode this problem as a *symbolic transition system* and then solve it with the help of our *constraint solver*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossing of the river.  We use the following variables:
* $\texttt{F}i$ for $i\in\{0, \cdots, n\}$ is the number of farmers on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{W}i$ for $i\in\{0, \cdots, n\}$ is the number of wolves on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{G}i$ for $i\in\{0, \cdots, n\}$ is the number of goats on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0, \cdots, n\}$ is the number of cabbages on the western shore after the 
  $i^{\textrm{th}}$ crossing.

## Auxiliary Functions

The following functions generate the *formulas* required by the constraint solver. This ensures that the logical expressions can be evaluated dynamically using the variable names.

The invariant guarantees that there is no problem on either shore of the river. There is no problem if on the shore where there is no farmer, the following conditions hold:
* If the wolf is on that shore, the goat must be on the opposite shore.
* If the goat is on that shore, the cabbage has to be on the opposite shore.

In [ ]:
function invariant(F: Variable, W: Variable, G: Variable, C: Variable): Formula {
    return `your code here`;
}

The transition formula defines a valid river crossing:
* The farmer travels in every crossing.
* She takes at most one of her goods with her in the boat.
* If the farmer travels from the western shore to the eastern shore, the number of goods on the western shore have to decrease, otherwise they have to increase.

In [ ]:
function transition(F𝛼: Variable, W𝛼: Variable, G𝛼: Variable, C𝛼: Variable, 
                    F𝛽: Variable, W𝛽: Variable, G𝛽: Variable, C𝛽: Variable): Formula 
{
    const eastToWest = [`your code here`,
                        `your code here`];
    
    const westToEast = [`your code here`, 
                        `your code here`];
    
    return `your code here`;
}

The function `wgcCSP` creates a CSP that tries to solve the problem with `n` crossings.

In [ ]:
function wgcCSP(n: number): CSP {
    const Vars    = "your code here";
    const Values  = "your code here";
    const Constrs = ["your code here",             // Start
                     "your code here",             // Goal
                     "your code here",             // Invariant
                     "your code here"]             // Transition Relation
    return [Vars, Values, Constrs];
}

In [ ]:
wgcCSP(3);

The function `findSolution` computes a solution to the problem by iteratively increasing the number of crossings `n`.

In [ ]:
function findSolution(): [number, Assignment] {
    let n = 1;
    while (true) {
        console.log(`Checking n = ${n}...`);
        const csp = wgcCSP(n);
        const result = solve(csp);
        if (result != null) { return [n, result]; }
        n += 2;
    }
}

In [ ]:
console.time("findSolution"); 
const result = findSolution(); 
console.timeEnd("findSolution");
result

## Visualizing the Result

In [ ]:
function showSolution(solution: Assignment, n: number) {
    for (let i = 0; i <= n; i++) {
        const F = Number(solution.get(`F${i}`));
        const W = Number(solution.get(`W${i}`));
        const G = Number(solution.get(`G${i}`));
        const C = Number(solution.get(`C${i}`));
        
        const leftSide = '🧑‍🌾'.repeat(F) + '🐺'.repeat(W) + '🐐'.repeat(G) + '🥦'.repeat(C);
        const rightSide = '🧑‍🌾'.repeat(1 - F) + '🐺'.repeat(1 - W) + '🐐'.repeat(1 - G) + '🥦'.repeat(1 - C);
        const gap = " ".repeat(28 - (F + W + G + C) * 2);
        
        console.log(`${leftSide}${gap}${rightSide}`);
        
        if (i < n) {
            const nextF = Number(solution.get(`F${i+1}`));
            const nextW = Number(solution.get(`W${i+1}`));
            const nextG = Number(solution.get(`G${i+1}`));
            const nextC = Number(solution.get(`C${i+1}`));
            
            if (F == 1) {
                const WB = W - nextW;
                const GB = G - nextG;
                const CB = C - nextC;
                console.log(' '.repeat(10) + `>>>  🧑‍🌾${'🐺'.repeat(WB)}${'🐐'.repeat(GB)}${'🥦'.repeat(CB)}  >>>`);
            } else {
                const WB = nextW - W;
                const GB = nextG - G;
                const CB = nextC - C;
                console.log(' '.repeat(10) + `<<<  🧑‍🌾${'🐺'.repeat(WB)}${'🐐'.repeat(GB)}${'🥦'.repeat(CB)}  <<<`);
            }
        }
    }
}

In [ ]:
const [n, solution] = result;
showSolution(solution, n);

In [ ]:
import * as tslab from "tslab";

function show_animated_solution(solution: Assignment, n: number) {
    if (!solution) {
        console.error("No solution provided.");
        return;
    }

    // Prepare the state data for the frontend JavaScript
    const states = [];
    for (let i = 0; i <= n; i++) {
        states.push({
            F: Number(solution.get(`F${i}`)),
            W: Number(solution.get(`W${i}`)),
            G: Number(solution.get(`G${i}`)),
            C: Number(solution.get(`C${i}`))
        });
    }

    // Generate a unique ID suffix to avoid element collisions if run multiple times
    const uid = Math.random().toString(36).substring(2, 9);

    const htmlContent = `
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-${uid}" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-${uid}" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-${uid}" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-${uid}" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of ${n}</span>
        </div>
        
        <svg width="600" height="250" style="background-color: #e0f7fa; border: 1px solid #b2ebf2; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="250" fill="#a5d6a7" />
            <rect x="450" y="0" width="150" height="250" fill="#a5d6a7" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Western Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Eastern Shore</text>

            <text id="west-f-${uid}" x="75" y="80" font-size="28" text-anchor="middle"></text>
            <text id="west-w-${uid}" x="75" y="120" font-size="28" text-anchor="middle"></text>
            <text id="west-g-${uid}" x="75" y="160" font-size="28" text-anchor="middle"></text>
            <text id="west-c-${uid}" x="75" y="200" font-size="28" text-anchor="middle"></text>
            
            <text id="east-f-${uid}" x="525" y="80" font-size="28" text-anchor="middle"></text>
            <text id="east-w-${uid}" x="525" y="120" font-size="28" text-anchor="middle"></text>
            <text id="east-g-${uid}" x="525" y="160" font-size="28" text-anchor="middle"></text>
            <text id="east-c-${uid}" x="525" y="200" font-size="28" text-anchor="middle"></text>

            <g id="boat-${uid}" style="transition: transform 1.5s ease-in-out;">
                <rect x="160" y="160" width="80" height="40" rx="8" fill="#8d6e63" stroke="#5d4037" stroke-width="2"/>
                <text id="boat-p-${uid}" x="200" y="188" font-size="22" text-anchor="middle"></text>
            </g>
        </svg>
    </div>

    <script>
    (function() {
        const states = ${JSON.stringify(states)};
        let currentStep = 0;
        let isAnimating = false;
        
        const boat = document.getElementById('boat-${uid}');
        const boatP = document.getElementById('boat-p-${uid}');
        const stepLabel = document.getElementById('label-step-${uid}');
        
        const elements = {
            west: {
                F: document.getElementById('west-f-${uid}'),
                W: document.getElementById('west-w-${uid}'),
                G: document.getElementById('west-g-${uid}'),
                C: document.getElementById('west-c-${uid}')
            },
            east: {
                F: document.getElementById('east-f-${uid}'),
                W: document.getElementById('east-w-${uid}'),
                G: document.getElementById('east-g-${uid}'),
                C: document.getElementById('east-c-${uid}')
            }
        };
        
        const icons = { F: '🧑‍🌾', W: '🐺', G: '🐐', C: '🥦' };
        
        // Web Audio API context for the motor sound
        let audioCtx;
        function playMotorSound() {
            if (!audioCtx) audioCtx = new (window.AudioContext || window.webkitAudioContext)();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            const osc = audioCtx.createOscillator();
            const filter = audioCtx.createBiquadFilter();
            const gain = audioCtx.createGain();
            
            osc.type = 'sawtooth';
            osc.frequency.value = 12; 
            filter.type = 'lowpass';
            filter.frequency.value = 150;

            gain.gain.setValueAtTime(0, audioCtx.currentTime);
            gain.gain.linearRampToValueAtTime(0.6, audioCtx.currentTime + 0.2); 
            
            osc.connect(filter);
            filter.connect(gain);
            gain.connect(audioCtx.destination);
            osc.start();
            
            return { osc, gain };
        }
        
        function stopMotorSound(sound) {
            if (sound) {
                sound.gain.gain.linearRampToValueAtTime(0, audioCtx.currentTime + 0.4);
                setTimeout(() => sound.osc.stop(), 400);
            }
        }

        function renderStaticState(step) {
            const s = states[step];
            elements.west.F.textContent = icons.F.repeat(s.F);
            elements.west.W.textContent = icons.W.repeat(s.W);
            elements.west.G.textContent = icons.G.repeat(s.G);
            elements.west.C.textContent = icons.C.repeat(s.C);
            
            elements.east.F.textContent = icons.F.repeat(1 - s.F);
            elements.east.W.textContent = icons.W.repeat(1 - s.W);
            elements.east.G.textContent = icons.G.repeat(1 - s.G);
            elements.east.C.textContent = icons.C.repeat(1 - s.C);
            
            boatP.textContent = '';
            boat.style.transform = s.F === 1 ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of ${n}';
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const fPass = Math.abs(s0.F - s1.F);
            const wPass = Math.abs(s0.W - s1.W);
            const gPass = Math.abs(s0.G - s1.G);
            const cPass = Math.abs(s0.C - s1.C);
            
            if (s0.F === 1) { 
                elements.west.F.textContent = icons.F.repeat(s0.F - fPass);
                elements.west.W.textContent = icons.W.repeat(s0.W - wPass);
                elements.west.G.textContent = icons.G.repeat(s0.G - gPass);
                elements.west.C.textContent = icons.C.repeat(s0.C - cPass);
            } else { 
                elements.east.F.textContent = icons.F.repeat((1 - s0.F) - fPass);
                elements.east.W.textContent = icons.W.repeat((1 - s0.W) - wPass);
                elements.east.G.textContent = icons.G.repeat((1 - s0.G) - gPass);
                elements.east.C.textContent = icons.C.repeat((1 - s0.C) - cPass);
            }
            
            boatP.textContent = icons.F.repeat(fPass) + icons.W.repeat(wPass) + icons.G.repeat(gPass) + icons.C.repeat(cPass);
            
            const sound = playMotorSound();
            boat.style.transform = s1.F === 1 ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1500));
            
            stopMotorSound(sound);
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-${uid}').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });

        document.getElementById('btn-prev-${uid}').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });

        document.getElementById('btn-reset-${uid}').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    `;

    tslab.display.html(htmlContent);
}

In [ ]:
show_animated_solution(solution, n);